# Phase 11: Complete Fraud Model Comparison

**Phase:** 20

Aggregate all completed phase artifacts into one comparison report.

This notebook preserves the project contract established in Phase 1 and Phase 2:

- Permanent code in GitHub
- Permanent dataset and artifacts in Google Drive
- ULB `creditcard.csv`
- Chronological 60/20/20 partitions
- Training-only preprocessing
- Validation-only model and threshold selection
- One final test evaluation
- Reproducible artifacts and acceptance checks

> Research POC only. Do not use the outputs for live transaction decisions.


## Storage and execution model

- GitHub repository: `deepak-malviyaa/enterprise-adaptive-learning-lab`
- Dataset: `MyDrive/enterprise-adaptive-learning-lab-data/raw/creditcard.csv`
- Runs: `MyDrive/enterprise-adaptive-learning-lab-data/outputs/runs/`
- Colab `/content`: temporary runtime only


In [ ]:
# Common setup: Drive, GitHub, project installation, and paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, random, shutil, subprocess, hashlib, platform

SEED=42
random.seed(SEED)
REPO_URL='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
DATA_PATH=DRIVE_ROOT/'raw'/'creditcard.csv'
RUNS_ROOT=DRIVE_ROOT/'outputs'/'runs'
RUNS_ROOT.mkdir(parents=True,exist_ok=True)

if (PROJECT_ROOT/'.git').exists():
    result=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
else:
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
print(result.stdout); print(result.stderr)
if result.returncode!=0: raise RuntimeError('Git operation failed.')

pyproject=PROJECT_ROOT/'pyproject.toml'
text=pyproject.read_text(encoding='utf-8').replace('>=3.10,<3.13','>=3.10,<3.14')
pyproject.write_text(text,encoding='utf-8')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
if not DATA_PATH.exists(): raise FileNotFoundError(f'Missing permanent dataset: {DATA_PATH}')
print('Project and dataset ready.')


In [ ]:
# Common data contract, checksum, chronological split, and preprocessing
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,brier_score_loss,confusion_matrix

np.random.seed(SEED)
def sha256_file(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()

DATASET_SHA256=sha256_file(DATA_PATH)
df=pd.read_csv(DATA_PATH)
EXPECTED=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED)-set(df.columns))
if missing or set(df.Class.unique())!={0,1}: raise ValueError(f'Invalid dataset. Missing={missing}')

df=df.sort_values('Time',kind='mergesort').reset_index(drop=True)
df['row_id']=np.arange(len(df))
a=int(len(df)*.60); b=int(len(df)*.80)
train_df=df.iloc[:a].copy(); val_df=df.iloc[a:b].copy(); test_df=df.iloc[b:].copy()
FEATURES=['Time']+[f'V{i}' for i in range(1,29)]+['Amount']
X_train=train_df[FEATURES].copy(); y_train=train_df.Class.to_numpy()
X_val=val_df[FEATURES].copy(); y_val=val_df.Class.to_numpy()
X_test=test_df[FEATURES].copy(); y_test=test_df.Class.to_numpy()
scaler=StandardScaler().fit(X_train[['Time','Amount']])
for frame in [X_train,X_val,X_test]: frame.loc[:,['Time','Amount']]=scaler.transform(frame[['Time','Amount']])
X_train=X_train.to_numpy(np.float32); X_val=X_val.to_numpy(np.float32); X_test=X_test.to_numpy(np.float32)
print('Checksum:',DATASET_SHA256); print(X_train.shape,X_val.shape,X_test.shape)

def metrics(y,p,threshold=.5):
    pred=(p>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'pr_auc':float(average_precision_score(y,p)),'roc_auc':float(roc_auc_score(y,p)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'brier':float(brier_score_loss(y,p)),'threshold':float(threshold),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def select_threshold(y,p):
    rows=[]
    for t in np.arange(.01,1,.01): rows.append(metrics(y,p,float(t)))
    return max(rows,key=lambda r:(r['f1'],r['recall']))


In [ ]:
# 5. Discover latest run for each phase and aggregate available metrics
patterns={'logistic':'*_phase1_fraud_logistic','xgboost':'*_phase2_fraud_xgboost','mlp':'*_phase3_fraud_mlp','swiglu':'*_phase4_fraud_swiglu','transformer':'*_phase5_fraud_transformer','moe':'*_phase6_fraud_moe','decision_env':'*_phase7_decision_environment','rl':'*_phase8_rl_baselines','grpo':'*_phase9_grpo','tabulargrpo':'*_phase10_tabulargrpo'}
rows=[]
for model,pattern in patterns.items():
    runs=sorted(RUNS_ROOT.glob(pattern),reverse=True)
    if not runs: continue
    run=runs[0]; metric_file=run/'test_metrics.json' if (run/'test_metrics.json').exists() else run/'metrics.json'
    if metric_file.exists():
        data=json.loads(metric_file.read_text()); rows.append({'model':model,'run':run.name,**{k:v for k,v in data.items() if isinstance(v,(int,float,str))}})
comparison=pd.DataFrame(rows); display(comparison)
RUN_DIR=RUNS_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_phase11_model_comparison'); RUN_DIR.mkdir(parents=True); comparison.to_csv(RUN_DIR/'all_models.csv',index=False); (RUN_DIR/'summary.md').write_text('# Complete Fraud Model Comparison\n\n'+comparison.to_markdown(index=False),encoding='utf-8')


## Interpretation rule

Prediction models and decision policies optimize different objectives. PR-AUC and F1 compare predictors. Mean reward and action constraints compare decision policies. They must not be ranked in one column without explaining the objective.
